# Geração de 10 cenas: silo + superfícies + posições do sensor

Este notebook usa três arquivos:

- `catalogo_superficies.yaml`: catálogo das 45 superfícies;
- `config_sensor.yaml`: configuração do TMF8829 e intervalos permitidos de pose;
- `config_silos.yaml`: geometria física dos silos (corpo, teto, tremonha e saída).

Nesta etapa são geradas **10 cenas determinísticas**, sem aleatoriedade.

Cada cena combina:

```text
superfície
+
geometria física do silo
+
altura relativa do sensor
+
azimute
+
deslocamento radial
+
inclinação do eixo óptico
```

O silo é escolhido em um único ponto, `SILO_MODEL`. Trocar esse valor e
reexecutar o notebook reproduz as mesmas 10 configurações relativas,
adaptadas à geometria do novo silo.

A seed aleatória fica para a próxima etapa.

In [ ]:
from pathlib import Path

import numpy as np
import yaml
import matplotlib.pyplot as plt

CONFIGS_DIR = Path("../configs")
OUT_DIR = Path("../out")

SURFACES_YAML = CONFIGS_DIR / "catalogo_superficies.yaml"
SENSOR_YAML = CONFIGS_DIR / "config_sensor.yaml"
SILOS_YAML = CONFIGS_DIR / "config_silos.yaml"
OUTPUT_DIR = OUT_DIR / "cenas_geradas"

# cenas/                    -> vista 3D da cena + metadados
# analise_geometria_cenas/  -> 3D ao lado do corte vertical, para conferir
#                              a geometria (chapéu, banda do sensor, folgas)
CENAS_DIR = OUTPUT_DIR / "cenas"
ANALISE_DIR = OUTPUT_DIR / "analise_geometria_cenas"

# Único ponto de troca do silo.
# Todo o resto do notebook deriva a geometria deste modelo.
SILO_MODEL = "30-04-60"

# Folga mínima entre o centro do sensor e o chapéu do silo.
ROOF_CLEARANCE_M = 0.05

# sensor - metade superior do chapeu do cone, a partir do corte da tampa
#
# O topo do cone é cortado em roof_cut_fraction (1/10 nos CASP). Acima do
# corte nada existe para o simulador. A partir do corte, o sensor desce no
# máximo metade da altura do cone que sobrou:
#
#   sensor_height_fraction = 0.0  -> topo máximo, logo abaixo do corte
#   sensor_height_fraction = 1.0  -> meia altura do cone abaixo do corte
#
# Observação de campo (Arthur, 2026-09-18): na prática o sensor não fica
# tão baixo, então a banda foi reduzida de 3/4 para a metade superior do
# cone.
#
# Atenção: isso inverte o sentido de sensor_height_fraction em relação à
# primeira versão do notebook, onde 0.0 era a menor altura acima da ração.
SENSOR_BAND_FRACTION = 1.0 / 2.0

GRID_SIZE = 140

CENAS_DIR.mkdir(parents=True, exist_ok=True)
ANALISE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carregar os YAMLs de entrada

In [2]:
with SURFACES_YAML.open("r", encoding="utf-8") as f:
    superficies = yaml.safe_load(f)["superficies"]

with SENSOR_YAML.open("r", encoding="utf-8") as f:
    sensor_config = yaml.safe_load(f)

sensor = sensor_config["sensor"]
pose_ranges = sensor_config["pose_ranges"]

print(f"Superfícies: {len(superficies)}")
print(f"Sensor: {sensor['model']}")
print(pose_ranges)


Superfícies: 45
Sensor: tmf8829
{'sensor_height_fraction': {'min': 0.0, 'max': 1.0}, 'azimuth_deg': {'min': 0.0, 'max': 360.0}, 'radial_offset_fraction': {'min': 0.0, 'max': 1.0}, 'tilt_deg': {'min': -38.0, 'max': 38.0}}


## 2. Geometria física do silo

### Convenção global de coordenadas

A origem vertical fica na interface entre a tremonha e o corpo cilíndrico:

```text
y = 0  ->  interface tremonha / corpo cilíndrico

corpo cilíndrico:  0 <= y <= cylinder_height_m
teto:              cylinder_height_m <= y <= cylinder_height_m + roof_height_m
tremonha:          -hopper_height_m <= y <= 0
saída / vão:       -(hopper_height_m + outlet_height_m) <= y <= -hopper_height_m
```

Portanto:

```text
roof_top_y = cylinder_height_m + roof_height_m
bottom_y   = -(hopper_height_m + outlet_height_m)

roof_top_y - bottom_y = total_height_m
```

Essa igualdade é validada numericamente ao carregar **cada** silo do
catálogo, não apenas o selecionado.

### Corte do topo (boca da tampa)

Nenhum silo termina em ápice fechado: todos têm furo no topo para a tampa.
O chapéu é um **tronco de cone**, cortado em `roof_top_radius_m`.

`roof_height_reference` diz a que ponto a cota `roof_height_m` se refere:

| valor | significado |
|---|---|
| `apex` | cota do **ápice** do cone cheio. A boca fica abaixo, em `cylinder_height_m + (1 − roof_cut_fraction) × roof_height_m`. Acima dela fica a tampa, que completa a `total_height_m` do catálogo. |
| `opening` | cota da própria **boca** da tampa. |

Os CASP usam `apex`, e isso está provado pelos ângulos exatos do catálogo:
teto de **40.00°** medido até `r = 0` e tremonha de **60.00°** — o `-60` do
nome do modelo é o ângulo da tremonha. Se a cota fosse de uma boca truncada,
o ângulo não sairia redondo.

O Mombuca usa `opening`: `roof_top_radius_m = 0.305 m` foi medido junto com
`roof_height_m = 1.000 m`.

Como o corte de um cone é auto-semelhante, a mesma fração vale para o raio e
para a altura:

```text
roof_top_radius_m = roof_cut_fraction × radius_m
```

Para o Mombuca a fração é derivada da medida real (`0.305 / 1.150 = 0.2652`):
o chapéu tem 0.61 m de diâmetro no topo, medido em campo.

Para os CASP não temos a medida do furo, então adotamos **0.10** como
premissa — um corte pequeno, só o necessário para o furo da tampa.

**O teto físico que limita o sensor é o plano do corte, não o ápice.**
Acima do corte nada é modelado nem desenhado: a geometria útil termina ali.

Com essa convenção o corpo cilíndrico continua ocupando `0 <= y <= H`, que é
exatamente o domínio já usado pelas funções de superfície. Por isso basta
definir `R`, `H` e `H0` a partir do silo carregado para manter a
compatibilidade com a lógica atual das superfícies.

In [3]:
with SILOS_YAML.open("r", encoding="utf-8") as f:
    silos_config = yaml.safe_load(f)


def limites_verticais(geometry):
    """Topo estrutural e fundo do vão, usados na conferência da altura total.

    Para os CASP o topo estrutural é o ápice do cone cheio mais a tampa;
    para o Mombuca é a própria boca.
    """
    roof_top_y = (
        geometry["cylinder_height_m"]
        + geometry["roof_height_m"]
    )

    bottom_y = -(
        geometry["hopper_height_m"]
        + geometry["outlet_height_m"]
    )

    return roof_top_y, bottom_y


def boca_da_tampa(geometry):
    """Cota e raio da boca da tampa: o teto físico que limita o sensor.

    Retorna (opening_y, raio_boca, altura_do_cone), onde altura_do_cone é a
    altura do tronco de cone, do beiral até a boca.
    """
    corte = geometry["roof_cut_fraction"]
    referencia = geometry["roof_height_reference"]

    if referencia == "apex":
        altura_cone = (1.0 - corte) * geometry["roof_height_m"]
    elif referencia == "opening":
        altura_cone = geometry["roof_height_m"]
    else:
        raise ValueError(
            f"roof_height_reference inválido: {referencia!r}"
        )

    opening_y = geometry["cylinder_height_m"] + altura_cone
    raio_boca = geometry["roof_top_radius_m"]

    # o corte de um cone é auto-semelhante: a mesma fração
    # vale para o raio e para a altura
    esperado = corte * geometry["radius_m"]

    if abs(raio_boca - esperado) > 1e-3:
        raise ValueError(
            f"roof_top_radius_m={raio_boca:.4f} m não bate com "
            f"roof_cut_fraction={corte:.4f} x radius_m="
            f'{geometry["radius_m"]:.4f} m (esperado {esperado:.4f} m)'
        )

    return opening_y, raio_boca, altura_cone


def validar_silo(item, tolerancia_m=1e-6):
    """Confere roof_top_y - bottom_y == total_height_m e o corte do topo."""
    g = item["geometry"]

    roof_top_y, bottom_y = limites_verticais(g)

    altura_calculada = roof_top_y - bottom_y
    erro = abs(altura_calculada - g["total_height_m"])

    if erro > tolerancia_m:
        raise ValueError(
            f'{item["model"]}: roof_top_y - bottom_y = '
            f"{altura_calculada:.4f} m, mas total_height_m = "
            f'{g["total_height_m"]:.4f} m (erro {erro:.2e} m)'
        )

    boca_da_tampa(g)

    return altura_calculada


silos = {}

for item in silos_config["silos"]:
    altura = validar_silo(item)
    silos[item["model"]] = item

    g = item["geometry"]
    opening_y, raio_boca, altura_cone = boca_da_tampa(g)

    print(
        f'{item["model"]:<14}'
        f'raio={g["radius_m"]:.3f}  '
        f'corpo={g["cylinder_height_m"]:.3f}  '
        f'boca(r={raio_boca:.4f}, y={opening_y:.4f})  '
        f"cone={altura_cone:.4f}  "
        f"total={altura:.3f}  OK"
    )

if SILO_MODEL not in silos:
    raise KeyError(
        f"SILO_MODEL desconhecido: {SILO_MODEL!r}. "
        f"Disponíveis: {list(silos)}"
    )

silo = silos[SILO_MODEL]
geometry = silo["geometry"]

ROOF_STRUCT_TOP_Y, BOTTOM_Y = limites_verticais(geometry)
ROOF_OPENING_Y, ROOF_TOP_RADIUS_M, CONE_HEIGHT_M = boca_da_tampa(geometry)

R = geometry["radius_m"]
H = geometry["cylinder_height_m"]
H0 = 0.0

# Banda onde o sensor pode existir: da boca da tampa
# até 1/3 da altura do cone.
SENSOR_Y_TOP = ROOF_OPENING_Y - ROOF_CLEARANCE_M
SENSOR_Y_BOTTOM = ROOF_OPENING_Y - SENSOR_BAND_FRACTION * CONE_HEIGHT_M

if SENSOR_Y_BOTTOM >= SENSOR_Y_TOP:
    raise ValueError(
        f"banda do sensor degenerada em {SILO_MODEL}: "
        f"{SENSOR_Y_BOTTOM:.4f} m >= {SENSOR_Y_TOP:.4f} m "
        f"(ROOF_CLEARANCE_M grande demais para este cone)"
    )

print()
print(f'Silo selecionado: {SILO_MODEL} ({silo["source"]})')
print(f"R  = {R:.4f} m   (raio do corpo)")
print(f"H  = {H:.4f} m   (altura do corpo cilíndrico)")
print(f"H0 = {H0:.4f} m")
print()
print(f'corte do topo  = {geometry["roof_cut_fraction"]:.4f} '
      f'({geometry["roof_top_radius_source"]})')
print(f"boca da tampa  = r {ROOF_TOP_RADIUS_M:.4f} m, y {ROOF_OPENING_Y:+.4f} m")
print(f"altura do cone = {CONE_HEIGHT_M:.4f} m (beiral -> boca)")
print(f"topo estrutural= {ROOF_STRUCT_TOP_Y:+.4f} m "
      f'({geometry["roof_height_reference"]})')
print(f"fundo da saída = {BOTTOM_Y:+.4f} m")
print()
print(f"banda do sensor: {SENSOR_Y_BOTTOM:.4f} m .. {SENSOR_Y_TOP:.4f} m "
      f"(espessura {SENSOR_Y_TOP - SENSOR_Y_BOTTOM:.4f} m)")

36-02-60      raio=1.830  corpo=1.840  boca(r=0.1830, y=3.2220)  cone=1.3820  total=6.650  OK
30-04-60      raio=1.480  corpo=3.680  boca(r=0.1480, y=4.7977)  cone=1.1177  total=7.770  OK
30-4,5-60     raio=1.480  corpo=4.090  boca(r=0.1480, y=5.2077)  cone=1.1177  total=8.170  OK
36-03-60      raio=1.830  corpo=2.760  boca(r=0.1830, y=4.1420)  cone=1.3820  total=7.570  OK
36-3,5-60     raio=1.830  corpo=3.170  boca(r=0.1830, y=4.5520)  cone=1.3820  total=7.980  OK
ovos_mombuca  raio=1.150  corpo=2.500  boca(r=0.3050, y=3.5000)  cone=1.0000  total=6.250  OK

Silo selecionado: 30-04-60 (CASP)
R  = 1.4800 m   (raio do corpo)
H  = 3.6800 m   (altura do corpo cilíndrico)
H0 = 0.0000 m

corte do topo  = 0.1000 (cut_fraction)
boca da tampa  = r 0.1480 m, y +4.7977 m
altura do cone = 1.1177 m (beiral -> boca)
topo estrutural= +4.9219 m (apex)
fundo da saída = -2.8481 m

banda do sensor: 4.2389 m .. 4.7477 m (espessura 0.5089 m)


In [4]:
def roof_max_y(radial_distance_m):
    """Maior coordenada y fisicamente disponível nesse deslocamento radial.

    O teto que limita o sensor é o tronco de cone que termina na boca da
    tampa, não o ápice do cone cheio.

    Para r <= roof_top_radius_m o limite é a cota da boca.

    Para roof_top_radius_m < r <= radius_m o limite acompanha exatamente o
    segmento reto da parede do chapéu entre

        (radius_m,          cylinder_height_m)
        (roof_top_radius_m, ROOF_OPENING_Y)
    """
    r = abs(float(radial_distance_m))

    if r > R + 1e-9:
        raise ValueError(
            f"deslocamento radial {r:.4f} m fora do raio "
            f"do silo ({R:.4f} m)"
        )

    if r <= ROOF_TOP_RADIUS_M:
        return ROOF_OPENING_Y

    if R <= ROOF_TOP_RADIUS_M:
        return H

    fracao = (R - r) / (R - ROOF_TOP_RADIUS_M)

    return H + fracao * (ROOF_OPENING_Y - H)


def roof_max_radius(y_m):
    """Maior raio disponível nessa cota. É o inverso de roof_max_y.

    Acima da boca só existe o furo da tampa; abaixo do beiral, o corpo
    cilíndrico inteiro.
    """
    y = float(y_m)

    if y >= ROOF_OPENING_Y:
        return ROOF_TOP_RADIUS_M

    if y <= H:
        return R

    fracao = (y - H) / CONE_HEIGHT_M

    return R + fracao * (ROOF_TOP_RADIUS_M - R)


print(f"roof_max_y(0.0000) = {roof_max_y(0.0):.4f} m  (boca da tampa)")
print(f"roof_max_y({R:.4f}) = {roof_max_y(R):.4f} m  (beiral)")
print(f"roof_max_radius({ROOF_OPENING_Y:.4f}) = "
      f"{roof_max_radius(ROOF_OPENING_Y):.4f} m")
print(f"roof_max_radius({H:.4f}) = {roof_max_radius(H):.4f} m")

assert abs(roof_max_y(0.0) - ROOF_OPENING_Y) < 1e-12
assert abs(roof_max_y(R) - H) < 1e-12
assert abs(roof_max_radius(ROOF_OPENING_Y) - ROOF_TOP_RADIUS_M) < 1e-12
assert abs(roof_max_radius(H) - R) < 1e-12

for _r in np.linspace(ROOF_TOP_RADIUS_M, R, 9):
    assert abs(roof_max_radius(roof_max_y(_r)) - _r) < 1e-9

print("chapéu e inverso coerentes")


# Maior raio que a banda oferece: fica no seu limite inferior, já com a
# folga do chapéu descontada. É por ele que radial_offset_fraction é
# normalizado, para que 1.0 alcance de fato a borda da banda.
RADIAL_MAX_M = roof_max_radius(SENSOR_Y_BOTTOM + ROOF_CLEARANCE_M)

print()
print(f"raio máximo da banda = {RADIAL_MAX_M:.4f} m "
      f"({100 * RADIAL_MAX_M / R:.1f}% de R)")
print(f"raio no corte da tampa = {ROOF_TOP_RADIUS_M:.4f} m")

roof_max_y(0.0000) = 4.7977 m  (boca da tampa)
roof_max_y(1.4800) = 3.6800 m  (beiral)
roof_max_radius(4.7977) = 0.1480 m
roof_max_radius(3.6800) = 1.4800 m
chapéu e inverso coerentes

raio máximo da banda = 0.7544 m (51.0% de R)
raio no corte da tampa = 0.1480 m


## 3. Domínio do silo e funções de superfície

As funções de superfície continuam idênticas às do notebook de validação.
A única diferença é que `R` e `H` agora vêm do silo selecionado, em vez de
serem constantes fixas:

- raio `R = radius_m`;
- altura `H = cylinder_height_m`;
- `x` e `z` no plano horizontal;
- `y` como eixo vertical, com `y = 0` na base do corpo cilíndrico.

In [5]:
eixo = np.linspace(-R, R, GRID_SIZE)
X, Z = np.meshgrid(eixo, eixo)

RAIO = np.sqrt(X**2 + Z**2)
MASCARA_SILO = RAIO <= R


def rugosidade(x, z, lambda_m, beta_deg):
    beta = np.deg2rad(beta_deg)

    ub = x * np.cos(beta) + z * np.sin(beta)
    vb = -x * np.sin(beta) + z * np.cos(beta)

    return (
        0.62 * np.sin(2 * np.pi * ub / lambda_m)
        + 0.38 * np.cos(4 * np.pi * vb / lambda_m)
    )


def base_automatica(relevo):
    valores = relevo[MASCARA_SILO]
    minimo = np.nanmin(valores)
    maximo = np.nanmax(valores)

    return H / 2 - (minimo + maximo) / 2


def finalizar_superficie(relevo, parametros):
    nu = parametros["nu"]

    if nu == "auto":
        b = base_automatica(relevo)
    else:
        b = nu * H

    Y = (
        b
        + relevo
        + parametros["a0_m"]
        * rugosidade(
            X,
            Z,
            parametros["lambda_m"],
            parametros["beta_deg"],
        )
    )

    Y = np.clip(Y, 0, H)
    Y = np.where(MASCARA_SILO, Y, np.nan)

    return Y


In [6]:
def gerar_superficie(item):
    tipo = item["tipo"]
    p = item["parametros"]

    r = RAIO
    A = 0.4 * R

    if tipo == "Plana (assentada)":
        relevo = np.zeros_like(X)
        return finalizar_superficie(relevo, p)

    if tipo == "Monte lateral com cratera central":
        epsilon = p["epsilon"]
        alpha = p["alpha"]
        depth_factor = p["depth_factor"]
        theta = np.deg2rad(p["theta_deg"])
        phi = np.deg2rad(p["phi_deg"])

        cx = epsilon * R * np.cos(theta)
        cz = epsilon * R * np.sin(theta)

        distancia = np.sqrt((X - cx) ** 2 + (Z - cz) ** 2)

        monte = np.tan(phi) * np.maximum(0, alpha * R - distancia)
        cratera = -depth_factor * A + np.tan(phi) * r

        relevo = np.minimum(monte, cratera)

        return finalizar_superficie(relevo, p)

    if tipo == "Cratera de descarga":
        rho = p["rho"]
        depth_factor = p["depth_factor"]
        phi = np.deg2rad(p["phi_deg"])

        D = min(
            depth_factor * A,
            rho * R * np.tan(phi),
        )

        F = max(
            0,
            rho * R - D / np.tan(phi),
        )

        relevo = np.minimum(
            0,
            -D + np.tan(phi) * np.maximum(0, r - F),
        )

        return finalizar_superficie(relevo, p)

    if tipo == "Monte deslocado (boca lateral)":
        alpha = p["alpha"]
        c1, c2 = p["c"]
        phi = np.deg2rad(p["phi_deg"])

        distancia = np.sqrt(
            (X - R * c1) ** 2
            + (Z - R * c2) ** 2
        )

        relevo = np.tan(phi) * np.maximum(
            0,
            alpha * R - distancia,
        )

        return finalizar_superficie(relevo, p)

    if tipo == "Plano inclinado (escorregada)":
        k = p["k"]
        phi = np.deg2rad(p["phi_deg"])
        theta = np.deg2rad(p["theta_deg"])

        relevo = np.tan(k * phi) * (
            X * np.cos(theta)
            + Z * np.sin(theta)
        )

        return finalizar_superficie(relevo, p)

    if tipo == "Cratera excêntrica (canal de fluxo lateral)":
        epsilon = p["epsilon"]
        rho = p["rho"]
        depth_factor = p["depth_factor"]
        theta = np.deg2rad(p["theta_deg"])
        phi = np.deg2rad(p["phi_deg"])

        cx = epsilon * R * np.cos(theta)
        cz = epsilon * R * np.sin(theta)

        distancia = np.sqrt((X - cx) ** 2 + (Z - cz) ** 2)

        D = min(
            depth_factor * A,
            rho * R * np.tan(phi),
        )

        F = max(
            0,
            rho * R - D / np.tan(phi),
        )

        relevo = np.minimum(
            0,
            -D + np.tan(phi) * np.maximum(0, distancia - F),
        )

        return finalizar_superficie(relevo, p)

    if tipo == "Frente de avalanche (dois patamares)":
        h = p["h"]
        x0 = p["x0"]
        theta = np.deg2rad(p["theta_deg"])
        phi = np.deg2rad(p["phi_deg"])

        u = (
            X * np.cos(theta)
            + Z * np.sin(theta)
            - x0 * R
        )

        kappa = h * H / (2 * np.tan(phi))
        relevo = (h * H / 2) * np.tanh(u / kappa)

        return finalizar_superficie(relevo, p)

    if tipo == "Cone truncado (topo achatado)":
        h_t = p["h_t"]
        epsilon = p["epsilon"]
        theta = np.deg2rad(p["theta_deg"])
        phi = np.deg2rad(p["phi_deg"])

        cx = epsilon * R * np.cos(theta)
        cz = epsilon * R * np.sin(theta)

        distancia = np.sqrt((X - cx) ** 2 + (Z - cz) ** 2)

        relevo = np.minimum(
            h_t * R * np.tan(phi),
            np.tan(phi) * np.maximum(0, R - distancia),
        )

        return finalizar_superficie(relevo, p)

    if tipo == "Rathole (canal central)":
        rho_0 = p["rho_0"]
        d_star = p["d_star"]
        n = p["n"]
        nu = p["nu"]

        b = nu * H

        Y = (
            b
            - d_star * (b - H0)
            / (1 + (r / (rho_0 * R)) ** n)
            + p["a0_m"]
            * rugosidade(
                X,
                Z,
                p["lambda_m"],
                p["beta_deg"],
            )
        )

        Y = np.clip(Y, 0, H)
        Y = np.where(MASCARA_SILO, Y, np.nan)

        return Y

    raise ValueError(f"Tipo não implementado: {tipo}")


## 4. Conversão dos parâmetros normalizados da pose

O sensor só existe dentro do chapéu, do corte da tampa para baixo, e no
máximo até a **metade da altura do cone** que sobrou depois do corte — ou seja, o sensor vive na metade superior do chapéu.

Dentro dessa banda o **raio é o parâmetro livre** e a altura se adapta a
ele. É o contrário do que seria natural escrever, mas é o que corresponde à
geometria: quanto mais longe do eixo, mais baixo o chapéu desce, então o
teto disponível cai junto.

### Deslocamento radial

Normalizado pelo maior raio que a banda oferece, que fica no seu limite
inferior:

```text
RADIAL_MAX_M    = roof_max_radius(SENSOR_Y_BOTTOM + ROOF_CLEARANCE_M)
radial_offset_m = radial_offset_fraction × RADIAL_MAX_M
```

Assim `radial_offset_fraction = 1.0` alcança de fato a borda da banda, e não
fica preso ao raio do corte.

### Altura do sensor

O teto disponível é o chapéu **naquele raio**, limitado pelo corte da tampa:

```text
y_teto = min(SENSOR_Y_TOP, roof_max_y(radial_offset_m) − ROOF_CLEARANCE_M)
y_piso = SENSOR_Y_BOTTOM

sensor_y = y_teto − sensor_height_fraction × (y_teto − y_piso)
```

Ou seja:

```text
sensor_height_fraction = 0.0
→ o mais alto que o chapéu permite naquele raio

sensor_height_fraction = 1.0
→ no piso da banda, a meia altura do cone abaixo do corte
```

Perto do eixo (`r <= roof_top_radius_m`) o teto é o próprio plano do corte,
e a banda tem a espessura máxima. Na borda da banda a folga vertical fecha:
sobra um único ponto. Entre os dois extremos a banda encolhe linearmente,
acompanhando o cone.

> Atenção: `sensor_height_fraction` tem sentido invertido em relação à
> primeira versão do notebook, onde `0.0` era a menor altura acima da ração.
> Agora `0.0` é o topo.

A altura não depende mais do nível da ração — ela é puramente geométrica. O
nível da ração virou só um critério de validade.

### Poses inválidas

A pose é marcada como inválida, **sem clamp silencioso**, quando:

- a ração sobe a ponto de invadir a banda do chapéu
  (`sensor_y <= surface_y + min_range_m`); ou
- `y_teto` cai abaixo de `y_piso`, o que não deve acontecer por construção.

Nesses casos a cena é reportada, o sensor não é desenhado e a validação
final falha.

### Inclinação

`tilt_deg = 0` aponta verticalmente para baixo.

Valores positivos inclinam o eixo óptico na direção do centro do silo;
valores negativos inclinam para fora.

In [7]:
def valor_superficie_em(Y, x_m, z_m):
    ix = np.argmin(np.abs(eixo - x_m))
    iz = np.argmin(np.abs(eixo - z_m))

    valor = Y[iz, ix]

    if np.isnan(valor):
        indices = np.argwhere(MASCARA_SILO)
        distancias = (
            (X[MASCARA_SILO] - x_m) ** 2
            + (Z[MASCARA_SILO] - z_m) ** 2
        )
        indice = indices[np.argmin(distancias)]
        valor = Y[indice[0], indice[1]]

    return float(valor)


def calcular_pose(Y, pose):
    # O raio é o parâmetro livre: normalizado pelo maior raio da banda.
    radial_offset_m = pose["radial_offset_fraction"] * RADIAL_MAX_M

    # A altura se adapta: o chapéu naquele raio é o teto disponível,
    # limitado pelo plano do corte da tampa.
    roof_y = roof_max_y(radial_offset_m)

    y_teto = min(SENSOR_Y_TOP, roof_y - ROOF_CLEARANCE_M)
    y_piso = SENSOR_Y_BOTTOM

    sensor_y = (
        y_teto
        - pose["sensor_height_fraction"] * (y_teto - y_piso)
    )

    azimuth_rad = np.deg2rad(pose["azimuth_deg"])

    sensor_x = radial_offset_m * np.cos(azimuth_rad)
    sensor_z = radial_offset_m * np.sin(azimuth_rad)

    surface_y = valor_superficie_em(
        Y,
        sensor_x,
        sensor_z,
    )

    min_sensor_y = surface_y + sensor["min_range_m"]

    resultado = {
        "sensor_y_m": sensor_y,
        "surface_y_m": surface_y,
        "roof_y_m": roof_y,
        "min_sensor_y_m": min_sensor_y,
        "sensor_y_ceiling_m": y_teto,
        "radial_offset_m": radial_offset_m,
        "radial_max_m": RADIAL_MAX_M,
    }

    # Não deve acontecer por construção, mas sem clamp silencioso.
    if y_teto < y_piso - 1e-9:
        resultado["valid"] = False
        resultado["motivo"] = (
            f"banda fechada em r={radial_offset_m:.4f} m: "
            f"y_teto={y_teto:.4f} m < y_piso={y_piso:.4f} m"
        )

        return resultado

    # Se a ração invade a banda do chapéu, não existe pose possível.
    if sensor_y <= min_sensor_y:
        resultado["valid"] = False
        resultado["motivo"] = (
            f"ração alta demais: sensor_y={sensor_y:.4f} m <= "
            f"surface_y + min_range_m={min_sensor_y:.4f} m "
            f"(superfície em {surface_y:.4f} m, r={radial_offset_m:.4f} m)"
        )

        return resultado

    tilt_rad = np.deg2rad(pose["tilt_deg"])

    inward = np.array(
        [
            -np.cos(azimuth_rad),
            0.0,
            -np.sin(azimuth_rad),
        ]
    )

    downward = np.array([0.0, -1.0, 0.0])

    direction = (
        np.cos(tilt_rad) * downward
        + np.sin(tilt_rad) * inward
    )

    direction = direction / np.linalg.norm(direction)

    resultado["valid"] = True
    resultado["position"] = np.array(
        [sensor_x, sensor_y, sensor_z]
    )
    resultado["direction"] = direction

    return resultado

## 5. Dez cenas determinísticas

Escolhemos superfícies distribuídas entre os tipos do catálogo e poses
diferentes dentro dos intervalos definidos no YAML.

Todas as poses são **relativas** (frações e ângulos), então a mesma lista
vale para qualquer silo: ao trocar `SILO_MODEL` as 10 configurações
continuam iguais, só que resolvidas na geometria do novo silo.

Não existe seed nesta etapa.

In [8]:
cenas = [
    {
        "surface_id": 1,
        "sensor_height_fraction": 0.90,
        "azimuth_deg": 0.0,
        "radial_offset_fraction": 0.00,
        "tilt_deg": 0.0,
    },
    {
        "surface_id": 6,
        "sensor_height_fraction": 0.80,
        "azimuth_deg": 0.0,
        "radial_offset_fraction": 0.25,
        "tilt_deg": 10.0,
    },
    {
        "surface_id": 11,
        "sensor_height_fraction": 0.70,
        "azimuth_deg": 45.0,
        "radial_offset_fraction": 0.50,
        "tilt_deg": -10.0,
    },
    {
        "surface_id": 16,
        "sensor_height_fraction": 0.60,
        "azimuth_deg": 90.0,
        "radial_offset_fraction": 0.75,
        "tilt_deg": 20.0,
    },
    {
        "surface_id": 21,
        "sensor_height_fraction": 0.50,
        "azimuth_deg": 135.0,
        "radial_offset_fraction": 0.90,
        "tilt_deg": -20.0,
    },
    {
        "surface_id": 26,
        "sensor_height_fraction": 0.85,
        "azimuth_deg": 180.0,
        "radial_offset_fraction": 0.25,
        "tilt_deg": 38.0,
    },
    {
        "surface_id": 31,
        "sensor_height_fraction": 0.75,
        "azimuth_deg": 225.0,
        "radial_offset_fraction": 0.50,
        "tilt_deg": -38.0,
    },
    {
        "surface_id": 36,
        "sensor_height_fraction": 0.65,
        "azimuth_deg": 270.0,
        "radial_offset_fraction": 0.75,
        "tilt_deg": 0.0,
    },
    {
        "surface_id": 41,
        "sensor_height_fraction": 0.55,
        "azimuth_deg": 315.0,
        "radial_offset_fraction": 0.90,
        "tilt_deg": 15.0,
    },
    {
        "surface_id": 45,
        "sensor_height_fraction": 0.95,
        "azimuth_deg": 330.0,
        "radial_offset_fraction": 0.50,
        "tilt_deg": -15.0,
    },
]

for i, cena in enumerate(cenas, start=1):
    print(i, cena)


1 {'surface_id': 1, 'sensor_height_fraction': 0.9, 'azimuth_deg': 0.0, 'radial_offset_fraction': 0.0, 'tilt_deg': 0.0}
2 {'surface_id': 6, 'sensor_height_fraction': 0.8, 'azimuth_deg': 0.0, 'radial_offset_fraction': 0.25, 'tilt_deg': 10.0}
3 {'surface_id': 11, 'sensor_height_fraction': 0.7, 'azimuth_deg': 45.0, 'radial_offset_fraction': 0.5, 'tilt_deg': -10.0}
4 {'surface_id': 16, 'sensor_height_fraction': 0.6, 'azimuth_deg': 90.0, 'radial_offset_fraction': 0.75, 'tilt_deg': 20.0}
5 {'surface_id': 21, 'sensor_height_fraction': 0.5, 'azimuth_deg': 135.0, 'radial_offset_fraction': 0.9, 'tilt_deg': -20.0}
6 {'surface_id': 26, 'sensor_height_fraction': 0.85, 'azimuth_deg': 180.0, 'radial_offset_fraction': 0.25, 'tilt_deg': 38.0}
7 {'surface_id': 31, 'sensor_height_fraction': 0.75, 'azimuth_deg': 225.0, 'radial_offset_fraction': 0.5, 'tilt_deg': -38.0}
8 {'surface_id': 36, 'sensor_height_fraction': 0.65, 'azimuth_deg': 270.0, 'radial_offset_fraction': 0.75, 'tilt_deg': 0.0}
9 {'surface_id':

## 6. Visualização do silo, do sensor e do campo de visão

O silo é desenhado como estrutura de linhas para não esconder a ração nem o
sensor. O desenho termina no corte da tampa: acima dele não aparece nada,
justamente para não confundir a região onde o sensor pode existir.

- círculo da base e parede cilíndrica, em cinza;
- chapéu (tronco de cone) até o corte da tampa, em azul;
- círculo verde marcando o limite inferior da banda útil, a meia altura
  do cone abaixo do corte;
- tremonha inferior e saída, em cinza.

Do sensor:

- posição em vermelho;
- eixo óptico em preto;
- limites aproximados do FOV diagonal em vermelho.

Cada cena salva **dois** PNGs, em pastas separadas:

- `cenas/scene_XXX.png`: somente a vista 3D;
- `analise_geometria_cenas/comparacao_cena_XXX.png`: a vista 3D ao lado do
  corte vertical.

Nesta etapa ainda não fazemos ray casting nem geramos a matriz 8×8.

In [9]:
def base_perpendicular(direction):
    referencia = np.array([0.0, 1.0, 0.0])

    if abs(np.dot(direction, referencia)) > 0.95:
        referencia = np.array([1.0, 0.0, 0.0])

    u = np.cross(direction, referencia)
    u = u / np.linalg.norm(u)

    v = np.cross(direction, u)
    v = v / np.linalg.norm(v)

    return u, v


def desenhar_sensor(ax, pose_calculada):
    position = pose_calculada["position"]
    direction = pose_calculada["direction"]

    axis_length = min(
        0.90,
        sensor["max_range_m"],
    )

    endpoint = position + direction * axis_length

    ax.scatter(
        position[2],
        position[0],
        position[1],
        s=80,
        color="red",
    )

    ax.plot(
        [position[2], endpoint[2]],
        [position[0], endpoint[0]],
        [position[1], endpoint[1]],
        linewidth=2.5,
        color="black",
    )

    half_fov = np.deg2rad(
        sensor["fov_diagonal_deg"] / 2
    )

    u, v = base_perpendicular(direction)

    for angle in np.linspace(0, 2 * np.pi, 5)[:-1]:
        lateral = (
            np.cos(angle) * u
            + np.sin(angle) * v
        )

        ray_direction = (
            np.cos(half_fov) * direction
            + np.sin(half_fov) * lateral
        )

        ray_endpoint = (
            position
            + ray_direction * axis_length
        )

        ax.plot(
            [position[2], ray_endpoint[2]],
            [position[0], ray_endpoint[0]],
            [position[1], ray_endpoint[1]],
            linewidth=1.2,
            color="red",
        )

In [10]:
def desenhar_silo(ax, n_geratrizes=24, n_circulo=97):
    """Estrutura de linhas do silo selecionado."""
    r_parede = R
    r_boca = ROOF_TOP_RADIUS_M
    r_saida = geometry["outlet_radius_m"]

    y_beiral = H
    y_boca = ROOF_OPENING_Y
    y_tremonha = -geometry["hopper_height_m"]
    y_fundo = BOTTOM_Y

    angulo = np.linspace(0, 2 * np.pi, n_circulo)
    geratrizes = np.linspace(
        0,
        2 * np.pi,
        n_geratrizes,
        endpoint=False,
    )

    # Mesma ordem de eixos usada em plot_surface(Z, X, Y).
    def circulo(raio, y, **kwargs):
        if raio <= 0:
            return

        ax.plot(
            raio * np.sin(angulo),
            raio * np.cos(angulo),
            np.full_like(angulo, y),
            **kwargs,
        )

    def parede(r0, y0, r1, y1, **kwargs):
        for a in geratrizes:
            ax.plot(
                [r0 * np.sin(a), r1 * np.sin(a)],
                [r0 * np.cos(a), r1 * np.cos(a)],
                [y0, y1],
                **kwargs,
            )

    estrutura = dict(color="0.35", linewidth=0.9, alpha=0.6)
    chapeu = dict(color="tab:blue", linewidth=1.0, alpha=0.75)
    banda = dict(color="tab:green", linewidth=1.8, alpha=0.9)

    # corpo cilíndrico
    circulo(r_parede, 0.0, **estrutura)
    circulo(r_parede, y_beiral, **estrutura)
    parede(r_parede, 0.0, r_parede, y_beiral, **estrutura)

    # chapéu: tronco de cone do beiral até o corte da tampa.
    # Acima do corte nada é desenhado.
    parede(r_parede, y_beiral, r_boca, y_boca, **chapeu)
    circulo(r_boca, y_boca, **chapeu)

    # superfície translúcida do chapéu, para conferir a folga do sensor
    RAIO_TETO, ANGULO_TETO = np.meshgrid(
        np.array([r_boca, r_parede]),
        angulo,
        indexing="ij",
    )

    Y_TETO = np.interp(
        RAIO_TETO,
        [r_boca, r_parede],
        [y_boca, y_beiral],
    )

    ax.plot_surface(
        RAIO_TETO * np.sin(ANGULO_TETO),
        RAIO_TETO * np.cos(ANGULO_TETO),
        Y_TETO,
        color="tab:blue",
        alpha=0.12,
        linewidth=0,
        shade=False,
    )

    # limite inferior da banda útil do sensor (1/3 da altura do cone)
    circulo(roof_max_radius(SENSOR_Y_BOTTOM), SENSOR_Y_BOTTOM, **banda)

    # tremonha e saída
    parede(r_parede, 0.0, r_saida, y_tremonha, **estrutura)
    circulo(r_saida, y_tremonha, **estrutura)
    parede(r_saida, y_tremonha, r_saida, y_fundo, **estrutura)
    circulo(r_saida, y_fundo, **estrutura)


def desenhar_corte(ax, Y, pose_calculada, cena):
    """Corte vertical no azimute do sensor, com zoom no chapéu.

    A vista 3D do matplotlib não tem z-buffer real: o ponto do sensor é
    pintado por cima do cone e parece atravessá-lo. Este corte é a leitura
    sem ambiguidade de projeção.
    """
    az = np.deg2rad(cena["azimuth_deg"])

    s_lin = np.linspace(-R, R, 241)

    y_racao = np.array([
        valor_superficie_em(Y, s * np.cos(az), s * np.sin(az))
        for s in s_lin
    ])

    y_chapeu = np.array([roof_max_y(abs(s)) for s in s_lin])

    # ração e chapéu
    ax.fill_between(s_lin, BOTTOM_Y, y_racao, color="tab:blue", alpha=0.25)
    ax.plot(s_lin, y_racao, color="tab:blue", linewidth=1.4, label="ração")
    ax.plot(s_lin, y_chapeu, color="tab:blue", linewidth=2.0, label="chapéu")

    # parede do cilindro
    ax.plot([-R, -R], [0, H], color="0.35", linewidth=1.4)
    ax.plot([R, R], [0, H], color="0.35", linewidth=1.4)

    # plano do corte da tampa
    ax.plot(
        [-ROOF_TOP_RADIUS_M, ROOF_TOP_RADIUS_M],
        [ROOF_OPENING_Y, ROOF_OPENING_Y],
        color="tab:blue",
        linewidth=2.4,
    )
    ax.axhline(
        ROOF_OPENING_Y,
        color="tab:blue",
        linestyle=":",
        linewidth=0.9,
        alpha=0.7,
    )
    ax.text(
        -R,
        ROOF_OPENING_Y + 0.012 * CONE_HEIGHT_M,
        f"corte da tampa  y={ROOF_OPENING_Y:.3f}",
        fontsize=8,
        color="tab:blue",
    )

    # Banda útil do sensor: a região realmente admissível, limitada pelo
    # cone. Não é um retângulo - o teto disponível cai com o raio.
    s_banda = np.linspace(-RADIAL_MAX_M, RADIAL_MAX_M, 241)

    y_teto_banda = np.array([
        min(SENSOR_Y_TOP, roof_max_y(abs(s)) - ROOF_CLEARANCE_M)
        for s in s_banda
    ])

    ax.fill_between(
        s_banda,
        SENSOR_Y_BOTTOM,
        y_teto_banda,
        where=y_teto_banda >= SENSOR_Y_BOTTOM,
        color="tab:green",
        alpha=0.16,
    )
    ax.plot(s_banda, y_teto_banda, color="tab:green", linewidth=1.3)
    ax.axhline(
        SENSOR_Y_BOTTOM, color="tab:green", linestyle="--", linewidth=1.0
    )
    ax.text(
        -R,
        SENSOR_Y_BOTTOM - 0.06 * CONE_HEIGHT_M,
        f"banda do sensor  {SENSOR_Y_BOTTOM:.3f} .. {SENSOR_Y_TOP:.3f}"
        f"  |  raio até {RADIAL_MAX_M:.3f} m",
        fontsize=8,
        color="tab:green",
    )

    if pose_calculada["valid"]:
        position = pose_calculada["position"]
        direction = pose_calculada["direction"]

        s_sensor = (
            position[0] * np.cos(az)
            + position[2] * np.sin(az)
        )

        d_s = direction[0] * np.cos(az) + direction[2] * np.sin(az)
        d_y = direction[1]

        comprimento = min(0.90, sensor["max_range_m"])
        meio_fov = np.deg2rad(sensor["fov_diagonal_deg"] / 2)

        for angulo in (-meio_fov, 0.0, meio_fov):
            ds = d_s * np.cos(angulo) - d_y * np.sin(angulo)
            dy = d_s * np.sin(angulo) + d_y * np.cos(angulo)

            ax.plot(
                [s_sensor, s_sensor + comprimento * ds],
                [position[1], position[1] + comprimento * dy],
                color="black" if angulo == 0.0 else "red",
                linewidth=2.0 if angulo == 0.0 else 1.0,
            )

        ax.scatter(
            [s_sensor],
            [position[1]],
            s=70,
            color="red",
            zorder=5,
        )

        ax.annotate(
            f"sensor y={position[1]:.3f}\nfolga chapéu "
            f'{pose_calculada["roof_y_m"] - position[1]:.3f} m',
            xy=(s_sensor, position[1]),
            xytext=(0.03, 0.06),
            textcoords="axes fraction",
            fontsize=8,
            color="darkred",
            arrowprops=dict(arrowstyle="->", color="darkred", linewidth=1.0),
        )

    # zoom no chapéu, que é onde o sensor vive
    ax.set_xlim(-R * 1.04, R * 1.04)
    ax.set_ylim(
        H - 0.40 * CONE_HEIGHT_M,
        ROOF_OPENING_Y + 0.14 * CONE_HEIGHT_M,
    )

    ax.set_xlabel("deslocamento radial com sinal (m)")
    ax.set_ylabel("y (m)")
    ax.set_title("corte vertical no azimute do sensor", fontsize=10)
    ax.grid(alpha=0.25)

## 7. Gerar as 10 cenas

Cada cena salva:

```text
cenas_geradas/
    cenas/
        scene_XXX.png     vista 3D da cena
        scene_XXX.yaml    silo, superfície, pose e grandezas derivadas
    analise_geometria_cenas/
        comparacao_cena_XXX.png   3D + corte vertical
```

In [11]:
def montar_3d(ax, Y, pose_calculada):
    """Monta a vista 3D da cena no eixo dado."""
    ax.plot_surface(
        Z,
        X,
        Y,
        linewidth=0,
        antialiased=True,
        alpha=0.85,
    )

    desenhar_silo(ax)

    # Pose impossível: nada de sensor no gráfico.
    if pose_calculada["valid"]:
        desenhar_sensor(ax, pose_calculada)

    ax.set_xlabel("z (m)")
    ax.set_ylabel("x (m)")
    ax.set_zlabel("y (m)")

    ax.set_xlim(-R, R)
    ax.set_ylim(-R, R)
    ax.set_zlim(BOTTOM_Y, ROOF_OPENING_Y)

    ax.set_box_aspect(
        (2 * R, 2 * R, ROOF_OPENING_Y - BOTTOM_Y)
    )


def gerar_cena(scene_id, cena):
    item = next(
        superficie
        for superficie in superficies
        if superficie["id"] == cena["surface_id"]
    )

    Y = gerar_superficie(item)
    pose = calcular_pose(Y, cena)

    marca = "" if pose["valid"] else " | POSE INVÁLIDA"

    titulo = (
        f"Cena {scene_id:03d} | silo {SILO_MODEL} | "
        f'superfície {item["id"]:02d} - {item["tipo"]}{marca}'
    )

    # ---------------------------------------------- 3D + corte, lado a lado
    fig = plt.figure(figsize=(15, 8))

    ax = fig.add_subplot(1, 2, 1, projection="3d")
    montar_3d(ax, Y, pose)
    ax.set_title("vista 3D", fontsize=10)

    ax_corte = fig.add_subplot(1, 2, 2)
    desenhar_corte(ax_corte, Y, pose, cena)

    if not pose["valid"]:
        ax_corte.text(
            0.01,
            0.01,
            "POSE INVÁLIDA: " + pose["motivo"],
            transform=ax_corte.transAxes,
            color="red",
            fontsize=8,
        )

    fig.suptitle(titulo)
    fig.tight_layout()

    fig.savefig(
        ANALISE_DIR / f"comparacao_cena_{scene_id:03d}.png",
        dpi=140,
        bbox_inches="tight",
    )

    plt.close(fig)

    # ------------------------------------------------------ somente a 3D
    fig_3d = plt.figure(figsize=(9, 9))

    ax_3d = fig_3d.add_subplot(111, projection="3d")
    montar_3d(ax_3d, Y, pose)
    ax_3d.set_title(titulo, fontsize=10)

    if not pose["valid"]:
        ax_3d.text2D(
            0.01,
            0.01,
            "POSE INVÁLIDA: " + pose["motivo"],
            transform=ax_3d.transAxes,
            color="red",
            fontsize=8,
        )

    fig_3d.savefig(
        CENAS_DIR / f"scene_{scene_id:03d}.png",
        dpi=140,
        bbox_inches="tight",
    )

    plt.close(fig_3d)

    derived = {
        "x_m": None,
        "y_m": None,
        "z_m": None,
        "surface_y_m": float(pose["surface_y_m"]),
        "roof_y_m": float(pose["roof_y_m"]),
        "roof_clearance_m": float(ROOF_CLEARANCE_M),
        "radial_offset_m": float(pose["radial_offset_m"]),
        "radial_offset_max_m": float(pose["radial_max_m"]),
        "sensor_y_ceiling_m": float(pose["sensor_y_ceiling_m"]),
        "sensor_band_fraction": float(SENSOR_BAND_FRACTION),
        "sensor_band_top_y_m": float(SENSOR_Y_TOP),
        "sensor_band_bottom_y_m": float(SENSOR_Y_BOTTOM),
        "axis_x": None,
        "axis_y": None,
        "axis_z": None,
    }

    if pose["valid"]:
        derived["x_m"] = float(pose["position"][0])
        derived["y_m"] = float(pose["position"][1])
        derived["z_m"] = float(pose["position"][2])
        derived["axis_x"] = float(pose["direction"][0])
        derived["axis_y"] = float(pose["direction"][1])
        derived["axis_z"] = float(pose["direction"][2])

    metadata = {
        "scene_id": scene_id,
        "valid": bool(pose["valid"]),
        "silo": {
            "model": SILO_MODEL,
            "radius_m": float(R),
            "cylinder_height_m": float(H),
            "roof_height_m": float(geometry["roof_height_m"]),
            "roof_height_reference": geometry["roof_height_reference"],
            "roof_cut_fraction": float(
                geometry["roof_cut_fraction"]
            ),
            "roof_top_radius_m": float(ROOF_TOP_RADIUS_M),
            "roof_opening_y_m": float(ROOF_OPENING_Y),
            "cone_height_m": float(CONE_HEIGHT_M),
        },
        "surface": {
            "id": item["id"],
            "type": item["tipo"],
        },
        "sensor": {
            "model": sensor["model"],
            "pose": {
                "sensor_height_fraction": cena[
                    "sensor_height_fraction"
                ],
                "azimuth_deg": cena["azimuth_deg"],
                "radial_offset_fraction": cena[
                    "radial_offset_fraction"
                ],
                "tilt_deg": cena["tilt_deg"],
            },
            "derived": derived,
        },
    }

    if not pose["valid"]:
        metadata["motivo"] = pose["motivo"]

    with (
        CENAS_DIR / f"scene_{scene_id:03d}.yaml"
    ).open("w", encoding="utf-8") as f:
        yaml.safe_dump(
            metadata,
            f,
            sort_keys=False,
            allow_unicode=True,
        )

    return metadata


metadados = []

for scene_id, cena in enumerate(cenas, start=1):
    metadados.append(
        gerar_cena(scene_id, cena)
    )

invalidas = [m for m in metadados if not m["valid"]]

print(f"{len(metadados)} cenas geradas para o silo {SILO_MODEL}")
print(f"{len(invalidas)} cena(s) inválida(s)")
print(f"{CENAS_DIR}/scene_XXX.png + scene_XXX.yaml")
print(f"{ANALISE_DIR}/comparacao_cena_XXX.png")
print(OUTPUT_DIR.resolve())

10 cenas geradas para o silo 30-04-60
0 cena(s) inválida(s)
cenas_geradas/cenas/scene_XXX.png + scene_XXX.yaml
cenas_geradas/analise_geometria_cenas/comparacao_cena_XXX.png
/mnt/c/Users/migue/Downloads/silo-sensor-lab-demo/superficies_candidatas_review/cenas_geradas


## 8. Conferência das cenas

In [12]:
print(
    f"Silo {SILO_MODEL}: "
    f"R={R:.4f} m, H={H:.4f} m, boca={ROOF_OPENING_Y:.4f} m, "
    f"cone={CONE_HEIGHT_M:.4f} m"
)
print(
    f"Banda do sensor ({SENSOR_BAND_FRACTION:.4f} do cone): "
    f"{SENSOR_Y_BOTTOM:.4f} .. {SENSOR_Y_TOP:.4f} m"
)
print()

for cena in metadados:
    pose = cena["sensor"]["pose"]
    derived = cena["sensor"]["derived"]

    if not cena["valid"]:
        print(
            f'Cena {cena["scene_id"]:03d}',
            f'| superfície {cena["surface"]["id"]:02d}',
            f'| INVÁLIDA: {cena["motivo"]}',
        )

        continue

    print(
        f'Cena {cena["scene_id"]:03d}',
        f'| superfície {cena["surface"]["id"]:02d}',
        f'| azimuth={pose["azimuth_deg"]:5.1f}°',
        f'| radial={pose["radial_offset_fraction"]:.2f}'
        f'->{derived["radial_offset_m"]:.3f} m',
        f'| height={pose["sensor_height_fraction"]:.2f}',
        f'| tilt={pose["tilt_deg"]:6.1f}°',
        f'| y={derived["y_m"]:.3f} m',
        f'| teto={derived["roof_y_m"]:.3f} m',
    )

Silo 30-04-60: R=1.4800 m, H=3.6800 m, boca=4.7977 m, cone=1.1177 m
Banda do sensor (0.5000 do cone): 4.2389 .. 4.7477 m

Cena 001 | superfície 01 | azimuth=  0.0° | radial=0.00->0.000 m | height=0.90 | tilt=   0.0° | y=4.290 m | teto=4.798 m
Cena 002 | superfície 06 | azimuth=  0.0° | radial=0.25->0.189 m | height=0.80 | tilt=  10.0° | y=4.334 m | teto=4.764 m
Cena 003 | superfície 11 | azimuth= 45.0° | radial=0.50->0.377 m | height=0.70 | tilt= -10.0° | y=4.334 m | teto=4.605 m
Cena 004 | superfície 16 | azimuth= 90.0° | radial=0.75->0.566 m | height=0.60 | tilt=  20.0° | y=4.302 m | teto=4.447 m
Cena 005 | superfície 21 | azimuth=135.0° | radial=0.90->0.679 m | height=0.50 | tilt= -20.0° | y=4.271 m | teto=4.352 m
Cena 006 | superfície 26 | azimuth=180.0° | radial=0.25->0.189 m | height=0.85 | tilt=  38.0° | y=4.310 m | teto=4.764 m
Cena 007 | superfície 31 | azimuth=225.0° | radial=0.50->0.377 m | height=0.75 | tilt= -38.0° | y=4.318 m | teto=4.605 m
Cena 008 | superfície 36 | azim

## 9. Validação numérica final

Confere, cena a cena:

```text
surface_y_m < sensor_y_m < roof_y_m
```

mais a distância mínima do sensor, a folga do teto, a contenção radial e a
soma geométrica da altura total do silo.

In [13]:
g = geometry

altura_geometrica = ROOF_STRUCT_TOP_Y - BOTTOM_Y

print("Altura total pela soma geométrica:")
print(
    f'  corpo {g["cylinder_height_m"]:.4f}'
    f' + teto {g["roof_height_m"]:.4f}'
    f' + tremonha {g["hopper_height_m"]:.4f}'
    f' + saída {g["outlet_height_m"]:.4f}'
    f" = {altura_geometrica:.4f} m"
)
print(
    f'  total_height_m no YAML = {g["total_height_m"]:.4f} m'
    f' | erro = {abs(altura_geometrica - g["total_height_m"]):.2e} m'
)
print()

print("Corte do topo:")
print(
    f'  roof_cut_fraction {g["roof_cut_fraction"]:.4f}'
    f' x radius_m {R:.4f} = {g["roof_cut_fraction"] * R:.4f} m'
    f" | roof_top_radius_m = {ROOF_TOP_RADIUS_M:.4f} m"
)
print(
    f"  boca em y = {ROOF_OPENING_Y:.4f} m"
    f" | altura do cone = {CONE_HEIGHT_M:.4f} m"
    f" | banda = {SENSOR_Y_TOP - SENSOR_Y_BOTTOM:.4f} m"
)
print()

print("surface_y_m < sensor_y_m < roof_y_m, dentro da banda do chapéu")
print()

falhas = []

for cena in metadados:
    scene_id = cena["scene_id"]
    d = cena["sensor"]["derived"]

    if not cena["valid"]:
        falhas.append(f'cena {scene_id:03d}: {cena["motivo"]}')
        print(f"cena {scene_id:03d} | INVÁLIDA")

        continue

    surface_y = d["surface_y_m"]
    sensor_y = d["y_m"]
    roof_y = d["roof_y_m"]
    radial = d["radial_offset_m"]

    checagens = {
        "acima da ração": sensor_y > surface_y,
        "distância mínima": (
            sensor_y - surface_y >= sensor["min_range_m"] - 1e-9
        ),
        "abaixo do chapéu": sensor_y < roof_y,
        "folga do chapéu": (
            roof_y - sensor_y >= ROOF_CLEARANCE_M - 1e-9
        ),
        "dentro da banda": (
            SENSOR_Y_BOTTOM - 1e-9 <= sensor_y <= SENSOR_Y_TOP + 1e-9
        ),
        "abaixo do corte": sensor_y <= ROOF_OPENING_Y - ROOF_CLEARANCE_M + 1e-9,
        "abaixo do teto local": (
            sensor_y <= d["sensor_y_ceiling_m"] + 1e-9
        ),
        "raio disponível": radial <= d["radial_offset_max_m"] + 1e-9,
        "dentro da parede": radial <= R + 1e-9,
        "teto coerente": (
            abs(roof_y - roof_max_y(radial)) < 1e-12
        ),
    }

    ruins = [nome for nome, ok in checagens.items() if not ok]

    if ruins:
        falhas.append(f'cena {scene_id:03d}: {", ".join(ruins)}')

    print(
        f"cena {scene_id:03d}"
        f" | {surface_y:7.4f} < {sensor_y:7.4f} < {roof_y:7.4f}"
        f" | folga chapéu {roof_y - sensor_y:6.4f} m"
        f" | acima da ração {sensor_y - surface_y:6.4f} m"
        f" | r {radial:.4f}/{d['radial_offset_max_m']:.4f} m"
        f' | {"OK" if not ruins else "FALHA"}'
    )

print()

if falhas:
    for f in falhas:
        print("FALHA:", f)

    raise AssertionError(f"{len(falhas)} cena(s) reprovada(s)")

print(f"As {len(metadados)} cenas do silo {SILO_MODEL} são válidas.")

Altura total pela soma geométrica:
  corpo 3.6800 + teto 1.2419 + tremonha 2.3036 + saída 0.5445 = 7.7700 m
  total_height_m no YAML = 7.7700 m | erro = 0.00e+00 m

Corte do topo:
  roof_cut_fraction 0.1000 x radius_m 1.4800 = 0.1480 m | roof_top_radius_m = 0.1480 m
  boca em y = 4.7977 m | altura do cone = 1.1177 m | banda = 0.5089 m

surface_y_m < sensor_y_m < roof_y_m, dentro da banda do chapéu

cena 001 |  3.5061 <  4.2897 <  4.7977 | folga chapéu 0.5080 m | acima da ração 0.7837 m | r 0.0000/0.7544 m | OK
cena 002 |  1.5400 <  4.3338 <  4.7636 | folga chapéu 0.4298 m | acima da ração 2.7938 m | r 0.1886/0.7544 m | OK
cena 003 |  1.7438 <  4.3338 <  4.6054 | folga chapéu 0.2716 m | acima da ração 2.5900 m | r 0.3772/0.7544 m | OK
cena 004 |  1.9580 <  4.3022 <  4.4471 | folga chapéu 0.1450 m | acima da ração 2.3441 m | r 0.5658/0.7544 m | OK
cena 005 |  1.7894 <  4.2705 <  4.3522 | folga chapéu 0.0817 m | acima da ração 2.4811 m | r 0.6790/0.7544 m | OK
cena 006 |  1.7322 <  4.3101